# Blood Work & Clinical Nutrition Pipeline — Step-by-Step Guide

**What is this pipeline?**  
Raw medical lab reports are dense, jargon-heavy, and difficult for non-specialists to interpret.  
This notebook demonstrates a **Two-Stage Chained LLM Architecture** that:
1. **Stage 1 (Biomarker Triage):** Ingests raw lab text and extracts every biomarker, flagging it as `HIGH`, `LOW`, or `NORMAL` against standard clinical reference intervals.
2. **Stage 2 (Nutrition Synthesis):** Takes the structured output from Stage 1 and synthesizes an accessible health summary along with targeted dietary recommendations tailored to Indian nutritional patterns.

---

### End-to-End Pipeline Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                       INPUT REPORT                              │
│                    "blood_work.txt"                             │
│     (Raw Unstructured Text: CBC, Lipid Panel, Metabolic)        │
└────────────────────────────────┬────────────────────────────────┘
                                 │
                                 ▼
┌─────────────────────────────────────────────────────────────────┐
│             STAGE 1 — EXTRACTION & CLINICAL TRIAGE              │
│                                                                 │
│  System Persona : Medical Data Extraction Assistant             │
│  Transformation : Parse raw text ──► Structured Key-Value Rows  │
│  Classification : Compare values against clinical reference     │
│  Output Format  : - Test: value | Status: HIGH/LOW/NORMAL       │
└────────────────────────────────┬────────────────────────────────┘
                                 │
                                 ▼ (Structured Biomarker Context)
┌─────────────────────────────────────────────────────────────────┐
│            STAGE 2 — FUNCTIONAL NUTRITION SYNTHESIS             │
│                                                                 │
│  System Persona : Clinical Nutritionist (Indian Dietary Habits) │
│  Input Context  : Flagged abnormal markers (Lipids/Cholesterol) │
│  Task           : 1. Plain-English Patient Summary (4-5 lines)  │
│                   2. Dual-Column Diet Strategy:                 │
│                      • Foods to Avoid                           │
│                      • Foods to Prioritize                      │
└────────────────────────────────┬────────────────────────────────┘
                                 │
                                 ▼
┌─────────────────────────────────────────────────────────────────┐
│                    FINAL DELIVERABLE                            │
│  • Actionable Patient Guidance & Doctor Discussion Topics       │
│  • Evidence-based Functional Nutrition Plan                     │
└─────────────────────────────────────────────────────────────────┘
```


---
## Step 1 — Setup & Environment Configuration

Load API keys from `.env` and initialize the LLM backend.


In [17]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

True

---
## Step 2 — Ingest the Unstructured Lab Report

Read the raw blood test report from disk (`blood_work.txt`). The document contains real-world clinical panels including Complete Blood Count (CBC), Lipid Profile, Renal/Liver Function, and Fasting Glucose.


In [18]:
llm = ChatOpenAI(model="gpt-6-luna")

with open("blood_work.txt", "r") as f:
    blood_report = f.read()

print(blood_report[:200])

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5â€“17.5)
Hematocrit:        44%          


---
## Step 3 — Stage 1: Biomarker Extraction & Reference Triage

**Why separate extraction from diet planning?**  
Separating the task into two distinct stages prevents hallucination and ensures mathematical accuracy:
- The model in Stage 1 focuses **strictly on data extraction and bounds checking**.
- It compares each measured value against the documented reference range and assigns a status: `HIGH`, `LOW`, or `NORMAL`.


In [19]:
extraction_prompt = f"""
You are a medical data extraction assistant.

From the blood report below, extract ALL test values and classify each one as HIGH, LOW, or NORMAL 
based on the reference ranges provided in the report.

Format your response as:
- Test Name: value | Status: HIGH/LOW/NORMAL | Reference: range

Blood Report:
{blood_report}
"""

extraction_response = llm.invoke(extraction_prompt)
extracted_values = extraction_response.text

print("=== STAGE 1: EXTRACTED VALUES ===")
print(extracted_values)

=== STAGE 1: EXTRACTED VALUES ===
- Hemoglobin: 15.1 g/dL | Status: NORMAL | Reference: 13.5–17.5
- Hematocrit: 44% | Status: NORMAL | Reference: 41–53%
- WBC: 6.8 x10^3/uL | Status: NORMAL | Reference: 4.5–11.0
- Platelets: 220 x10^3/uL | Status: NORMAL | Reference: 150–400
- Total Cholesterol: 238 mg/dL | Status: HIGH | Reference: <200
- LDL Cholesterol: 162 mg/dL | Status: HIGH | Reference: <100
- HDL Cholesterol: 36 mg/dL | Status: LOW | Reference: >40
- Triglycerides: 188 mg/dL | Status: HIGH | Reference: <150
- Glucose (Fasting): 92 mg/dL | Status: NORMAL | Reference: 70–99
- HbA1c: 5.3% | Status: NORMAL | Reference: <5.7%
- Creatinine: 1.0 mg/dL | Status: NORMAL | Reference: 0.7–1.3
- eGFR: 82 mL/min | Status: NORMAL | Reference: >60
- ALT: 28 U/L | Status: NORMAL | Reference: 7–40
- AST: 25 U/L | Status: NORMAL | Reference: 10–40
- Bilirubin Total: 0.8 mg/dL | Status: NORMAL | Reference: 0.2–1.2


---
## Step 4 — Stage 2: Functional Nutrition & Diet Strategy Synthesis

**Chaining Stage 1 into Stage 2:**  
Now we pass the clean, structured biomarker summary (`extracted_values`) from Stage 1 as input to a specialized clinical nutritionist prompt.

The LLM synthesizes:
1. A concise, empathetic summary in plain English.
2. A focused, practical Indian diet plan with high-yield foods to avoid vs. foods to prioritize.


In [20]:
diet_prompt = f"""
You are a clinical nutritionist specializing in Indian dietary habits.

Based on the blood work analysis below, write:
1. A short health summary in 4-5 lines explaining the patient's condition in simple language
2. A short, practical Indian diet plan having only two sections (1) Foods to avoid (2) Foods to eat more of. 
   Do not include any other sections in diet plan.

Blood Work Analysis:
{extracted_values}
"""

diet_response = llm.invoke(diet_prompt)

print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(diet_response.text)

=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===
**Health summary**
- Your blood counts, fasting glucose and HbA1c are within the stated normal ranges.
- Kidney and liver test results are also within the stated ranges.
- Total cholesterol, LDL (“bad” cholesterol) and triglycerides are high; HDL (“good” cholesterol) is low.
- These cholesterol results are worth discussing with your doctor, who can advise on your overall heart risk and follow-up.

**Foods to avoid**
- Limit deep-fried foods, chips, namkeen, pakoras and fast food.
- Cut back on ghee, butter, cream, vanaspati, fatty meats and processed meats.
- Avoid or reduce sweets, sugary drinks, fruit juice and large portions of maida-based foods such as biscuits and white bread.
- Limit alcohol, especially with high triglycerides.

**Foods to eat more of**
- Choose oats, barley, whole-wheat roti and other whole grains; keep rice portions moderate.
- Include dal, chana, rajma, beans and plenty of vegetables regularly.
- Choose whole fruit r